# 🕵️ Solo Task — Data Detective
**Time: 15 minutes · Work alone · Cameras on, mics muted · Type `?` in chat if stuck**

The survey office sent us the **raw** Campus Wellbeing Survey file. Before anyone analyses it, a detective must find
**everything that is wrong** with it. Real data is *always* dirty. At least **10 problems** are hiding in this file.

**Your goals**
1. 🔍 Find as many problems as you can (a scoreboard in chat: *type your count when time is up*).
2. 🧹 Fix them in `df_clean`.
3. ✅ Run the auto-checker until it shows **8/8**.
4. 🧠 Decide what to do with *suspicious but possibly real* students (outliers are **not** the same as errors).

In [1]:
#@title ▶️ Setup — run this cell first (double-click to see the code) { display-mode: "form" }
import base64, gzip, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 30)

# The survey data is stored inside this notebook, so nothing needs to be downloaded.
_DATA = {
    "raw": "H4sIAAzivWoC/61dybLlxnHdO8L/wA+4elGZWeNWQ8jaaGGG1x0t8oluWyQV3WTY8tcblWPhXhS4EBekqXbjAqjKyuGck4kvP/387fsPP3349O3j759//O7zx+8f/3j/+Pnx8bv3x+f3L5+O/+c3748vx9/6x4f//PHnz18+/M/7+38/vvzt/f3v8gePbz7+9a/vn354//DNz3//8uHbj/94fPnxm08f//bh+/dvP33Uq+Yf//3j558+/PTp+/cP//XjX9Zf++bH77//+af3D99/+uH4P1+O2x23/vLhyzc/fn5/vP/vx+/lP//1X3739W9STunx25+/HDf88uVBD0yPf/vxy0/vf3tAf0uP/tYf6ZHe2vFPegAd/8I3fDR6Q7m8pPb4/cefPn719Tef+N3wAcN/Yhx/v74N/on+yMf/mn9Cb/Ao/a3oA+QRDwDr1fW47XwI4Cv5AeD4V5sPAG9VL+/j/ADHT3T/Cfn7NJ9qvsn8CTz+lY8/qmhPkCDHE+QHol+e3uqjHE+LxzNnvrzxCpRHzW8gVxPQ5vnn1fz3j5sTX9353sev4FuWq7Hk5/U7tuB3soOfH1iO1WvHo8PxBI1/bcxNKPOnjzW1F4DH77+ejw5xKcy/k49/5sMf/56XQUHeEXzkam8P1B+//Y+v//TnP3z99Vf2+t8f69j4r861w3Xt6rz1eCO9GsbJfMDefj50OxZqbh7y1WIMaRqP7h219rJ08879uKIc1x+POX9hXgi8bMcm1G5Xl4SbbZt7PReOeNv4UraELtcnXfrRT9eva9f56Ss/AchSZODnh0fLtnmU++MPP3x3/ML7508/fHeyfcy8+3PnUO1GjwNO47FnyPB43/1CPa7u03IO+ym8hIc5zOPTH7W8Nd2AupgfX76+BfKKzz0Q882kB+J4CTtA7XkZeBMGX3vclveAb57Y8Pq03qGPfxjet4db++4HsVu+EnntgF/82MO5bLKA89xWM5wE57WD06MfL1r40ecWDjk7Zj+l2btjTY/fy+1P+z+tdv4zn0AOHnR9gN7cctvYmE+ZZnL8g7zU8tbz1m2umvo9xPz8+F1en6Z7aHp3eXQC8zlkRzYfpvN7Wzk3fL3NtDo57NTszsOMDgifjC58Nk6bnbse/g6n9T1qsjXL+fD3n/5Pzqqvd+eXbPNPp8myqRT+M/a1aq2Z2uIqzsY2F6zoroutYVZHVavZCwy6sjVd8aJejpcM7bR1c7R4XLwPNTPMTNeU3Vux2+A38Z9IDV+CRZy3wWcD+CnyGm/yPG/2CuA2x3vuC3C4xMd8j2mxKDdnq5sHvifztvhk9mvE5dP9m+k0sx869cJjBkyz+iNin15idbvHLYFvOf2Evj+77bm7h+/zsJNuvAaw/U23M+TU02AD7tNz2Yu056Dpv1DYdWQ2wjI39zi8wI5rLFGXej45/+egDfwAcf769KfgVze6dx/5cawLO+/CWzF/9FFqhEyybaTF53X1G/Nvs+cqvJHzP5KtXarPaxdrTxxjZrIifgf7m2xE83wh1744rTXo6BonNiDOtQDY4U2HUv0Q9rZZt64RB93t8StMf62+Hqnutv3Y58yeBvj2KSJenbtHFvHOtvfV2YCRjU8C9xr0y/QB9gxpbMMuWwgvwbR/Nr1a2BrGND07AK1cb0Dny4/Vf8wH4bXnZZyxjHzrkWjrQIn4DeqS8XH6MH+1Njc+rPsTiFlD13KIxYjytD9LHWbK9vQTS94Hmc/QYY6cOXPm2dSUSvHUuePM+84p43EzYJNB8QDzXz2rBzjMWRfxOAHb3IUOd8WneJ5gkH2Q03As8BEXzJvW9uzL9CChZB7IR7CJA0LJwNojN98JoN0PZN2HHGeYM9h5vIoHM6rtInsOdzoDcePjTFHA8B97/YBYLkPCfIh5JOaBOtZSw1IlzeCrJ4GQ82qNcf+ieVCkESBreGRfxfII6nhdPRQuPcYST9WNsyXoDsBR/FwvICHnQDOU2HHml59vbhaAY5PC4IzC+txsu0WKprHa3uiLA40DNK9M/My8aUMX/MjerN4o7er4yaaLA6uRBmTNnYu/czquX298CsBZ6xWSu5dqOYynfdjPiVteji7NgDdfM3PqxuEbpG5D9oa2Z3S1cJZEFE4DpIJJHoOTZHNmND29hi/z4xbywfP+zlXnUUF7FDkcyLXVeY02/XhcPUNbnBvM9XkRlvhNnKpq/ST7UIYW3tVjya+QS6WnVYjiE4mdKPBO1MgI508extjt+D9lQmsaj3zQOCGbxx8sGnFEwqh+x6s1xkoAV6KFbUoNovPLzM3w+juvYXU9DWNJxmoYw3y5qN7z4cTeX1dBXiJxSiFlNEU8mplG9ngEo74mlfESE4fQjaj2AjPJK56XI+JLTjHz8nnJPJTEB5pPFFk4ru6ES7aKIp3S0Ma1a7Y3L5bRN3d+4zmViSjWNNxIFPNMYB6tKCKhtn0tVPgRmmZzy+KP5c2hXDjBSIoyZxRWy/JTDH2w4zSheaS8j0Iw7beyTwF1xCQpLcxoYlXZEY3XpFCWf4b+h5x8cf4U2aQaD6S2lCNLJtQVOiMzm6qrd1TA6dcymwr3SXjXXMpslyyjLI5f5AZ3ABhM7DFzTSwLOePY0DOUPahg7xsQrvAJbpzK5Cip5nGujmJlpE1MItAjnB0FzWJFx8N4NQL1FUG0urZwEOaQwGlQUwSmei6YxrMvj9fnEK5pAIgBQoBwVhnnbjDMegj5mLLzOlWT80B2t7xE9BJIBIMjhT6zVYE1e0U9vBpBgx5xOfuG28Lq9eb/iBCUoe2xM9RNi2OHVUvpIxJZ9Ehjn8dLCDscgIcwSBbCWuTAtLWaxpVg4vOjULggSIfVWA2ZyzX2CY1NlTj9rbHyjFxHBj7qCxgil2c+coMPH8Td5y92DxuQYYNfzXJHAKisAJQkQ3PfU8B+ZWt0DBAydAn2Aymb8y1x5tpF9I3jz2UfH39U/5urZnPVPXgutDcDWHDgJZfics5BZHhKY9eVREOFFhhR41fJXtPi2BTUUo5OVMzqeb4/CpJny5DqCUnwBZBVBHPgWUAE+V/N0Rw69mGXCWJhGLBwCpMjiSc5SU7B9Bs8qCxHKevliSHBSOUPNz4D+BL4GSnnlbfoiz3Ctx3AijuviewCJBfXDDLz+YEF/C7QXwxoPQSoGWTSrcsKxWRPAHOHPflz+CzUBADXIDifqPrq5Z6utw+aWh89shV9chLLXAPLgvOFG3uuxbtyKIyIMhQ2ETUrgp+qiadThFyDSgYq6UOTOnjM45TtJeiSSAE1YEEzwEPwXNejEMRfKf7k1DeIWOXomRdQibcgyxoag5bGLYfY1ZSsntESJy1cDlARJKQYkkdaezvvR8Y8OvNBsPPfAuRNB1g4/XD/nWfebgxGSxcQTpjvkbtlSWHRIUycNaA5rox3ri8LYOMHsBmE4wQYzArsNfeDaTIPwRKh+pX46D1CT75IN4GNVfZaaTdOtebpaRG4R4BO4WZJc1y0PEdytDbNxQIW0hbwFdgvDqqHvOKA2XG747anNW56HBfTSFo7h3fDVm/qVFKSexpnlqxFcLa8ULxJcpW386WSYiQ+mVkypG6r7StdIN3k2AyYTqvKcbqNoS0OumKhvZ/EpqA1+U8Uw9od8gRo+zWArhwz8b77MjIp44GaWnKOeQWcGWAqC1bSpERI5h/HlZHPGJE0FxWUgS2VktHT3Wt8qNscDR4P2apuOU4Tks4WjnDHUfCj8q0V4gLqhk+A7XuJquwJniXOjocW9uwgPLnpDg2mRjebP8Zw47M0MYOlJ14k47GC3869+3KOcQJQNj44mT2bp1izXoraEtN1cOBjzafH4FGtbGCmyBZZ8PXuI+CNzp4Wg2tSX1cZMbLglHdUZzGoMBtHy66AgqPt29SsemCBR4+jf6xoUExjXJFtkuB09XUcGdn4yLL84FqArLA617bAidnhovm/kskyuDZ3oqgA3WBLyLXIjKNOlmSHWYNqILiCWQFUWxIcGWZdvulBLLkeN/hY5iUUr4uyBGAcVI1QkcoTtrWSFMTORpn+ZojrNOAgXXPayGtQVlEop+p00cyXuieJ0NJVkBN5CSpJuYDreV5rZTmE4zz7bivCpS4Q8ysaTUqkZhe+x3AZyamLV8TFhBUppDF14z3yYnyg7geNG+hu/Zj6VWUzD/5QdL+Ey7brq+M7Obet+5sXa2XGMbB0I+rCei5YrkBFFBeQFWh2fNR/GrdypOdXrmuFCVAFMszwSLZRF5oK0wVcHxgVOlIN6odrfwEKcmknyp5fQRAyXBAyEo4aF66VVpocXmnywjh7NrKjgAJ73dMWgP6SYx53b0xNFeV6XeCk3JCFPkrX3g/JK3wynysB7agrDVtFPHndE81cmaQ2VK6mkKW558xP9Djf2GjEQHaE314ZLRjXoq6ijKaggTkQ/THPjKfk6fqpBczFhVqHhFpM1yAUj5T6rEeyjLiptzRXq9ECJhxqZTD1q0pciKnOITqzo+JARUYL57DVmm/rmKpQRHaX45RIC1gs3YIqqG4GPNvrhunVGm9S7pLOojkySYlAdmq44PW1zIk2mZuTM1FdEYbCyXm+8hS5jkvN88k2uu8AobUNzkTalJPT2bHRDy+JGFNLUVONnc8CLjo1Z9N4WxTLD10JEtwsHRifWCRnJSvHGZLC4BjHVl1Db+gsZTEzGJq6LqrSJ4YOz+RKd3RlKLueTGzlT3H4zh2/C7yBqAqZZK53qY3h8LtHuv+E6ANH3cQJH0gUVVqAlsKWMrzs33FXweJQxSEcddAAlTxCX3VDpnTViT2V9VkoPXdA6UlTeVKoyNKRA1MSvqcbcdNNpdyAOjjNbynMq/C1fRKTtvQdH3+5QhaEFhNktQWwPT1CW+I2Mh63lOZdcSSSYisHINemz/d0tVxLi4YKY0JfV5O6nh6E7nW2HBW+krHstYai4YFGFKj32e5QNtbVEE3oWJh5r5lOhOo1U6xOwpCFSpVy4CJOyo2u6GQLmUGGnPSJfU34at1jyd24Kz80Kioe03yt0n6SFyz+ZygRWNl/DVN3MpWLLuctdb8JxEl/U7zDTZ+XkLzoTO2WFEd2gov9kwb1IBWp9cszKMqerKfYcnbWFLMhOrFf0h2rjlVwA86bhvlhPkFBLeZygXmswOpQD1YMMipGLiQPJEft8JdN7TIY2GycvkEEUkZTojOgl21nwHCJuUVzJ5jCno4M7pQALwspMmlawpEk4jOIhmAVe7zBUwJqauFFXNBcaOxyuzzo6kR1FgrTShC7NDyC4YtG7LSNnQGEutRgHVXuc/hSIwnKNgoxEM/0SlbGUCKKlOAWiDrtzbl0DZwYiyhlZJ0IoOfRY5vFZ1Y4rJrJRob/eUqbD7ewppVLLk1Okyxy6yJEnTE8qd/R7DOSqk/hXGhUzUxLiIZz2lJ9IZxWwaQ1SoRsGxJsFZdJIzo5gnTkbeqVYgVzu/WL1XkSRZx1E9gteWr+jAKt2G9SrnFYk8OMiD2EQrXeKCxIy/HM784BFewdcvLEotULGCrxOSiP5AwbknHkHpBplEvZukGP6OWQUGuhNoPc7twYity0LSq/ktWkS4i1Afd6s6q5uJaCEL0ysEpUer59jplHPqSzSQRvYBoJiKQI7pKi7NERHZAxS65ONSGkS7XBdKKduVLL7Qtabh5SBaRLotNKknYiejWprgtLlEu/Eftww5Sx9kMrRFDRC41oGxrXtD0HMIYk0TW/SfnGeabsMG4l2FwJKih1Cq1p5nguNel71mkwaySUu6lmu6a3R5ZjCXKN2HrODoaiagHIgYluD2fgYpk7WEmkLinsuWnPYPC+qZVbWHIoreJJjpSoIt0wOojoBAueVKDWvFNlCciFs93PJbVbLsr6WAQVTdp+UONcU96/AgiyC/YK2bwKKMRv53rgdSuJcArFOjAwWFeaGjDzqKXekt8q13bidhj3TZHn3ejYGSJs2g0hGGONLhCjByreUFNS7Yt8AZcGSJhn0kql0W5+QZAG6Uuqz79g5kz1toXSeA6Kis2aAcsiRew7KTqLHk5Kano9U4dzusF+ih4LkFDXjAdnxMXVDCVtZKQsgmsqgisBuHAXaFAFvd84aJKmyKrsrtCVWV3ksg4VTuliHG0RIhBDN6pFYbCfJFSaQcArqX3m8puqsiiUSVk8pHcG4r4EEuBUAJjiZj0PZ3b/ktt1kOHOZfVu6qAzWE+jJ5xHsrETJTO7oHLepu7V+baQFAO8bKPgT509U44OuW5aiBINNuW2uSsp62UHShsbs7CJtgf5gu1dS/HyeFQu3/ly1fTWydlYxlNvC7jBuwB8LGGF0jjWe5tOv+2Sqd5u1CR9IAdSQmrfroWVZMpYBeHQZMlt0WSOLRqguhwR5S69GVWIK0v7875VQEnzxr5N6W9rmCye7mRK+/6MqmikN/eC5r3RaASFrkwJk9KPT0U4SeXnLXK4gSI0ZRdRrMqzKIQTzl6P3UFIHCSzSh8cP+SDWb05ZVUYwlncwjDuzLh52cnqlu7UO5a2KXk6H6OyyAqHHoHoqocCGwQZtO5FDyjabpEX4pZgGwyGBiRp7mpW68zMN3psCfAuIkmjirengwkDtWPFbAfx7hRDEQaPrNeler4bMGShW201d5KzGUw0nu0Pq6Zr0WoCvV+GNnGp7aH5qkS1bGmKI8lH5Xf7Hl1p6Owxobk+vjuFhxcaAkbMmYQ0b1id1Ymyr99GEznE0uBVI9Vs8wWKAZvtSR45mUuRdc9ah0EL1l3MlCJHi3ilRZUNC4nBvgtMom+dzt1bvBPiRhncPBFYCTgUtZKd+8mBvIi70BtJTnxnO9XZUHeMI/JKlaUlqJuOMJLqSzXmGjWGoriGF9WqtfZS5NW+7QkR/DM/RKQi7Jl1JVQ//JTKPhXjitvAgmxFomz5khhT37m+zHwtvraql5nCWBaV4LbcFi50LKJONOVDCO1owL65xdjEKuINb7BkOMDhQ8h3reLTlzHYEhCkit/quhYTAz3M/4nHSopmC5DM8RvNky40VoGtxFKMD8IRG4fY3REjwgY85H5/K9LA0jmSLjcjE1Jy2d4KmGTXk6pyiIzMqIFaPp+htbt+MGpcXGzHynrvRTk2bqP70JJMMvB2Yt57tGLVl4YKkcCCqqlls9HidY/UMe/tpTNe3fj8UiCVM3NdmjnmVvt8h3M/kXTCymmVlvoaMz1M0I2bWUJVY3ZMsWnWiQdhZ3BZCApSLDJJ0R1I4p0C6vYCaGzlek0bw5MtoGqtVrlexnJbSCbefHkLFfQ3MNo15hmNsSPhMiPDwv2JoDip4OXIupw8bXdOFMYyXEGHIxStIuJHjnN3lTc27YiMrDfcjqv6cR3Kcwr1SdfQGBeysQyrSnRcXsyIOHtNF/NXU45Eso9wq30AO/KmmKHXYVL1dgt10EjXKKjKK9c+OAGcYdw1tk2hNCdNaovaXo8zaTAecOB9X8ujc4qzIJUMBZhyqpQL2eUaQJpHMe1L7AbUBUDU6yXk7lIaXKTlE1NpztrkvJ+RIBNlSiC1yhtFN17O5W75GrdIF+POup0jHg7gaAi0694O8QdV+2rBx5JJixnNStRkGOMaZzT2rXpbIUcuTn1nTKTIXXvb1BGmxk9BBosZlbVDql7UEes6ZIcaKYAlRg9DQ7cilbSK9bk7M7l+uetksrJgMXWnva4+mswLyNBChH65ltchKeRHmdvscxhwXiI3lRscA5OK71CoMgO5hXu12J8vJrQwzaykaVpb29aUP41y01c0tN5IZ8qwnmTPyZUUX53TlsQ5AyhXEgnAWDR/Kfebjp7jAZqPttIF7M1Ei5F2XcNQlsSSS5nU/nUdVkwS67iRXzZtzyvRG6QTJnA6dKtCR7v0IaJAzHwC2wpm9dlm4rnMcvGJ9OTnl0EtmvzFaI4ebfWwpzqYJ2gqianGUzy3F2K78WN1Uc+n4P6ayNgsgyW8JhoUwDVAVbJnVO64e1KS8Ia6RJvSISoEsElxXeYT5n8eBEvj4hiGqrLowQWOCs678RkZgYPVy/vPh1cDkGQwq4A2JjwQ3LXH8ag0VnY+UP0AVAeEQ9W5Dvqj51ZV1Klbls/Vol0VxwJaGVTrbSiRes4dOWllG6pMaHknDeU6RO4uulCyWR0tIIh7X0iaWnNumwPUTUsXUxrpcg+yUk068gcNS6pSSnpaNDbTQopqMEJPqhAOzKzGYtDYA6FYRMwDhoj3UIb5jILbFteikLaTzy7tik4cKv2k6aPXMR1NW7F0dJZqsdsCh2Fu9wI3LpLkOSwm8QSMEIJcq7MEGARNKEBNGYcBg9HMUHfUAOjMFgODycDg5jKA1MeNO7SERqMKOTMCC4ePJ4HgIgZC/ovgIhB0dxq8yOg79h2GtsIt016c+492eRjX1JKYQFFeRizApb3RKF3zdk4s6ZDXrPFY+1h4zpP5gCc05YRmd5+bJqG52rCMiMeQaK+mBWuDa6syPIRo0Ou+GpAxGZ37gIY1NSo/6drGROmqsDrVpDapKykGu4yYKekUiM/KGYEzmmTBfv+oIxLCflhRd7tDy0LAhZXRytLgZlZr02mDdK5GejDdu1GdWbcOvJZBa11cxmPQbjhIZiwIdD6xp0DcLpBcrkD1+u6SRBdTQzuqMGOocdNHJfTtbsxnYtyaT2lXV3PYqyWxue7mksHEcbu+tdRg0nsliIJLfvqGTRZD7frcybPvug7b7NcWkxVjXEZDdJ2HtBTyF7RVzOSxsmuZkMgtXx4vqbc7wZwkfrik4DUbarno1+/7KJJPGjSPNWRMFp95q4L7rv0wKYGWnAAFb4sv0Q+CW+KE9W3av7eICrQKMRi51KdNEMka+WitHswhzxnFWMP0C8IzGe8r2kvXz/IQ05j22Oum9lSe1tEHxcBpWlD9VcZqEPa7+TDjaV4sGg9evPqgsuU/m5rhooJu5vEDxz2uP5VPT0hwxNuyDAle3X5J5S771jkvXdHkHOLPshQgOC45ZFuIqtuIy4ByJfXNB2GzdgI8QWFNKXDbBR5uCHKtpa6pP7660R6KXCy79y2hhrCsZTrf5+lgyUY5edJtrdCLFv+pfWUJemBt9qKvQgu688dKTPobG7/NhUtRoV0Tx91juLT3Hu+wn2ouS/tvhh1/VrzGaPSCmx8oy7yDRTfKwayEdnzcTQjV8QM25Fow7WEzWtx6Cty2slRtJCT3x5kUVy9BLNy15Hi2IaCyDytSFXj+FWcF0bhAhJqew3QalMHwRMiXE+4E4NULcMuYiznThqHgxov5D0NF6+R0jou/a4opic/z2cSFp4daD9YYSdZm5mCx52bMtxJCqI33I9p/66I0JWqvDXjGmcf7os3lO8o0r1PwlsmF7v7XuxC9+SFknjldoz8gaXZd3O9YNN+O/oxfYANQB21aLB2oESw7EYJP+ENwSiTzhQz/WlKRsuCAiDc1l05JGgunLjyogA+WPafdvGeeRMgyTx60VmwdmI/JUfnfikm4jXpo7aVBaIQWy/PQukGx1gmXKPld9q9sxAyJ0l+PHqQlm7PhcsUcWQx8bXStZuD5utrOyf4LvCE4xn/UcprAsKKXXS2wrxkAzUTSZwu2beispvwTZ4E1nHhdGkoR8j6JkM8EUBiQLVwcwjRu55Qn/Q2d+e0Tx2dcighOiW6BUFq+0wHLrNVVTZBK3Y3d7artAB9xqNzwOgcij6WXaklkWeyjLZnalWsy4VCG4IDXJCD7mHfz+d00rdFMDqjiA3va6vC9pS2KuNHSQVyes/9T8RQ9rEUCLz/KNN3ya4z9gDE2cT8r87bAJMl653ycKuBV1fDMQtNSOXSKwWiW+9Z+o98EYeCaQQ7DIA9Gw2s0Y46dFGdw25QNtsVlcFGfJKhP/ij71DXphCv52IVUkabBDA0H9Isghgtqjec5+8OBdyywJ1CKST9isiD4gNhQsKZ27bqb6pjF7mR6zFC8McbPpHqvh+JJFzFBAoxSNxbOOvjSuENfs89JxkWXxYmABTHKv6BrEAhWxm4Li2AffIgvJuSazqXAaVho4+QXY/5XtcnfeZn3N/bzTKookKOmN61zhehszXf4bdP+Yjx9NIIHgdhCtDsEtWvnh0AC8qmdrJF0aZ7J9EvNWO1Bwe0bklijHit9R0vzRJ1zrwFYY1V8RSW/tKc/ze1FHzRBIbNrok9wjfx4hRa6wtDp/L0gnt++dCinHaqhI7dRZQmcCrSY5Obvn3d4WvOe1GGEpn/ryuNQ2X62h56qKht8t3w8Y+TtZAaDtmjppCtkEpfQJYy80ynx4FQD0jE0MkO09YbsNRut0E96dKFhks2b1VmV/LkD/3xNf1V2Nh9FZsk0WUd3jyszrCMFLIcCURC35VMd3ArhtNHNF3ewL62jfYERuXKyGji9DkIQ6hD0rJn/9cHi8bWSXPOWtSIfb60DlBwACuA2jxv+GMoi5auWjZw+lZTgZsCjzsJAU3EvzGE6dc7iXcuhDWYQQbikXdUKaopJk3CXEmg5hksXqZZjZf1sUEE3u7WKKlaJcgBCb+Rs7nfzti8pjlxyHt8+kNdiBCJsv7Axg5bi0EJ6ZsWBYqICpXxLoFtLDqyfyeAubssF89Mna2BFkqpP4tJmQzLuqEbLZ8ONilWGRrZlxLm6rbJ88ylDv8TSbXhVfN9QVZ1tpnQG49drRR1p+SIcgFiOCUyjP4/aTn/rwIHjl/KlJ/m+m7NO951EoBMAqovBKIUg0rVksJVhBRwBZ+fFZUT/dVjXRG0/CssGGFB8ZI75kZgSO3abj8Z8j5XyrMs8qly2Q7iKn9iqde8w4ipUSJm2t27ue+uTeCcmjGeiW78hWfxpzjiUFEiod3DcpPH8aRQdVFtXORULi80M66bTFWwaGQaK2ZIewB7fucG8hXKLk0FNRYDGr4UXThn3b1C1qyBmXYMp6YPAzKltP5OKLBxIoeVHU0bHVw4w/UITTHd7GNHKUkSH41OFdl8pQPvWUPFYnH0cin83Cyi91lI8yUTbUaSQyZoH9BydCHUrwOkKecak72ahdPla29P6r1tIY2ERcB1JA8swRcJ8kzTLZLmYgCuDthVMMTJu37Vujfc6JpujeDFKdOlMyZvWWJs8KR+aEippmUbpsw/GimVYDpa1kqyq+Sk6Uyi6OJDKZnJFPTX05G7SrZjkg23/tSPhQId2+4eSkz/9sFQbl4M7Bj94U+nA8oEFdqL2yqM9DeBZO5q7Al9jjX20JICAN+0rSKrYiUGGUCNbN7uhX5QiGwLh2zZxRYrV/ye/9EmpvY4jO08A0fAl3kumOYoe2TMovEmBjI2F85e+spS+Dv/1+Oyn5T7o/cio0d8nJeRwnXk3C4qnINvwXzApQJFOBCd+rken8BB8cNkFVYOwoncJQ4L73EhRfHZwjdFNJIM37bn73mcZ9KTycRnXMWywQCBxedsFZrOiKUTE2b4L1P7pkeWEu++SWLua9n2eBDstxiZjv2SuZVAKaJFteQuaUi660OEosv+ycZbcvqhFT7GBNUU+cmrV5tLK8zy5UNGA+DQM2qD40G8/l51Pn+STH6hmOjbOskEMUthqtaTzmVarRetsWbo3yAHg1WMObxrwrvGuDTyhk8tHzvz0kdynmVvpPC3qqWjtd59m44LFZkc2Q125EySkL6Ofu7j608jmpo04OXQfzMiFy8e7D3OwaJz9Blnlod8kw4lA+xSEcjO4lzu67IPD4rxH1Y9kUrRRXuS/poCR3kOD2Qi1fyS0jrnlW9Vt9QEAsAwW4cLFBYt1OzeMCRqVbpjw1UdAxyRHuPs+SVEGM8d25viyhX81oV5PREz+8YG+9CHwPEQjAUa6zBu6l1ya8oF902y4ZDq/dNCv6RI8NN3VKUflBfdItPmu5fC5D6jzvkwzErgTpg3G13R8Zry1Ct/68lV2IHjxfiW5++sKsBpU4jFv+Tpy3X8bSTohBXCAGBgH8mFCW3ksr+iq9X2AlCoYjTM8OM+16vjagylPTsrSZaNcdHwqLVBFHvVO79GUcojP2/YUgiGfs9X2ELcMTGtL070aLhOWVjMfPvAPf/7j8RJ/+Pc//fmP509zDT+9TBZ08x9Z5sU51rltXgKdRLO0YBtWvHy7AS4nWEPSL0IJd4TnbzJGE2THm8Zp6UPH9ZtyaDB/pN4wtt9pRkVss7VO+FdGq4/PoEp3+gdteKFVQ0WG/4QGjurVKkQPlDePmPBskRA+DUM9e0AKuNzOYk7Rk+wdQH0zycBmKEjxgbAWENLS+/9WGCM/TIUAAA=="
}

def load(name="clean"):
    """load('raw') -> messy survey file | load('clean') -> cleaned survey file"""
    return pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(_DATA[name]))))

print("✅ Setup complete. Available datasets:", list(_DATA))

✅ Setup complete. Available datasets: ['raw']


## Step 0 · Load the raw file

In [2]:
raw = load("raw")
print(raw.shape)
raw.head()

(530, 13)


,student_id,program,year,age,residence,study_hours_week,sleep_hours,caffeine_cups_day,social_media_hours_day,part_time_job_hours_week,commute_minutes,stress_score,exam_score
0,CS-0400,Business,3,20,Hostel,18.0,8.8,0,0.7,0.0,13.0,2.2,73.2
1,CS-0507,Data Science,2,19,Hostel,19.0,6.9,0,0.8,4.0,9.0,3.1,58.5
2,CS-0449,Business,1,19,Hostel,16.7,8.0,1,4.0,0.0,11.0,7.2,71.6
3,CS-0489,Data Science,1,18,Hostel,11.0,7.3,2,1.8,0.0,12.0,4.3,62.5
4,CS-0014,Business,4,22,Hostel,10.6,5.1,2,3.4,0.0,7.0,2.5,64.1


## Step 1 · Detective toolkit
| Question | Tool |
|---|---|
| How big? What types? What's missing? | `raw.info()` · `raw.isna().sum()` |
| Any impossible numbers? | `raw.describe()` · `raw.sort_values("col")` · `raw[raw["age"] > 60]` |
| Duplicates? | `raw.duplicated().sum()` · `raw[raw.duplicated(keep=False)]` |
| Messy categories? | `raw["program"].value_counts()` |
| Is missingness random? | `raw.groupby(raw["sleep_hours"].isna())["part_time_job_hours_week"].mean()` |

## Step 2 · Hunt!  *(each cell is a workspace — edit and re-run freely)*
### 2a. Numbers that cannot be real

In [3]:
raw.describe().round(1)

,year,age,study_hours_week,sleep_hours,caffeine_cups_day,social_media_hours_day,part_time_job_hours_week,commute_minutes,stress_score,exam_score
count,530.0,530.0,530.0,481.0,530.0,530.0,530.0,530.0,517.0,530.0
mean,2.5,20.8,20.5,6.8,1.9,2.9,6.8,37.4,5.8,66.8
std,1.1,12.7,43.9,1.4,1.6,1.2,8.4,34.8,2.0,42.4
min,1.0,5.0,1.0,-7.5,0.0,0.3,0.0,4.0,1.0,26.7
25%,2.0,19.0,12.4,6.1,1.0,2.2,0.0,12.0,4.6,57.1
50%,2.0,20.0,18.4,6.7,2.0,3.0,0.0,25.0,5.9,65.6
75%,3.0,21.0,24.0,7.4,3.0,3.7,14.0,52.0,7.2,73.7
max,4.0,250.0,999.0,25.0,8.0,6.8,30.0,224.0,10.0,999.0


In [4]:
# Which rows have impossible values? Look at the min/max above, then filter. Examples:
raw[raw["age"] > 60]
# Try the same idea for: age < 15, study_hours_week, sleep_hours, exam_score ...

,student_id,program,year,age,residence,study_hours_week,sleep_hours,caffeine_cups_day,social_media_hours_day,part_time_job_hours_week,commute_minutes,stress_score,exam_score
241,CS-0412,Engineering,2,199,Hostel,19.8,7.1,7,3.3,0.0,15.0,4.7,55.3
515,CS-0131,data science,4,250,Home,23.8,7.5,4,2.4,0.0,32.0,5.0,68.4


### 2b. Duplicates

In [5]:
# How many exact duplicate rows are there?
raw.duplicated().sum()

np.int64(10)

### 2c. Messy categories

In [6]:
raw["program"].value_counts()

program
Engineering      135
Business         133
Data Science     115
Design            72
Biz               13
DS                12
design            10
data science       9
business           7
BUSINESS           6
engineering        5
Data Sci           3
Eng.               3
Data Science       2
Design             2
 Engineering       2
ENGINEERING        1
Name: count, dtype: int64

### 2d. Missing values

In [7]:
raw.isna().sum()

student_id                   0
program                      0
year                         0
age                          0
residence                    0
study_hours_week             0
sleep_hours                 49
caffeine_cups_day            0
social_media_hours_day       0
part_time_job_hours_week     0
commute_minutes              0
stress_score                13
exam_score                   0
dtype: int64

### 2e. Is the missingness *random*? (the real detective question)
If `sleep_hours` is missing for **random** students, we can ignore it. If it's missing for a **specific kind** of student, our analysis is biased.

In [8]:
# Compare the average job hours of students who skipped the sleep question vs. those who answered.
# Hint: raw.groupby(raw["sleep_hours"].isna())["part_time_job_hours_week"].mean()

### 2f. Suspicious… but are they errors?

In [9]:
raw[raw["study_hours_week"].between(40, 100)].sort_values("study_hours_week", ascending=False)

,student_id,program,year,age,residence,study_hours_week,sleep_hours,caffeine_cups_day,social_media_hours_day,part_time_job_hours_week,commute_minutes,stress_score,exam_score
403,CS-0477,Engineering,3,21,Hostel,61.0,5.8,0,3.2,26.0,11.0,9.6,70.6
203,CS-0183,Data Science,4,22,Hostel,58.0,7.7,2,3.2,0.0,20.0,9.6,65.8
327,CS-0109,Data Science,3,21,Commuter,55.0,7.4,1,1.1,0.0,52.0,9.6,69.2


In [10]:
# Also look at the best-scoring students who barely study:
raw[raw["exam_score"].between(90, 100)].sort_values("study_hours_week").head(5)

,student_id,program,year,age,residence,study_hours_week,sleep_hours,caffeine_cups_day,social_media_hours_day,part_time_job_hours_week,commute_minutes,stress_score,exam_score
418,CS-0187,Design,3,20,Home,2.0,7.4,1,1.2,0.0,28.0,3.0,96.5
511,CS-0390,Design,2,19,Home,10.8,6.9,1,1.8,16.0,19.0,5.9,92.1


**Think:** a student who studies 58 h/week with stress 9.6 — a *typo*, or a real burned-out grinder?
A Design student with 2 study hours and an exam score of 96.5 — a *typo*, or a genuinely gifted student?
👉 **Write your decision and your evidence in the box below.**

## Step 3 · Evidence board 📝
*(double-click this cell to edit)*

| # | Problem found | How I found it | My fix |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| … | | | |

**Outlier decision (keep or remove? why?):** …

## Step 4 · Clean it
### 🧪 Auto-checker (run this cell, don't edit it)

In [11]:
REAL_OUTLIERS = ["CS-0183", "CS-0477", "CS-0109", "CS-0187"]   # (a hint: the checker knows some students are real!)
PROGRAMS = {"Engineering", "Data Science", "Business", "Design"}

def check_my_work(d):
    ok = lambda c: "✅" if c else "❌"
    def within(col, lo, hi):
        s = d[col].dropna()
        return bool(s.between(lo, hi).all())
    checks = [
        ("no duplicate students", not d["student_id"].duplicated().any()),
        ("program has exactly 4 clean labels", set(d["program"].dropna().unique()) == PROGRAMS and d["program"].notna().all()),
        ("age is realistic (15–60 or blank)", within("age", 15, 60)),
        ("study_hours is realistic (0–100 or blank)", within("study_hours_week", 0, 100)),
        ("sleep_hours is realistic (0–14 or blank)", within("sleep_hours", 0, 14)),
        ("exam_score is between 0 and 100 (or blank)", within("exam_score", 0, 100)),
        ("real extreme students were KEPT", d["student_id"].isin(REAL_OUTLIERS).sum() == 4 and
            d.loc[d["student_id"].isin(REAL_OUTLIERS), ["study_hours_week", "exam_score"]].notna().all().all()),
        ("you did not delete too much (≥ 505 rows left)", len(d) >= 505),
    ]
    for name, passed in checks:
        print(ok(passed), name)
    n = sum(p for _, p in checks)
    print(f"\n{n}/{len(checks)} checks passed" + ("  🎉 Clean!" if n == len(checks) else "  — keep going"))

print("Checker ready.")

Checker ready.


In [12]:
df_clean = raw.copy()

# 1) remove exact duplicate rows
#    hint: df_clean = df_clean.drop_duplicates()

# 2) standardise the program labels so only 4 remain.
#    Look at raw["program"].value_counts() above and add every variant you saw.
program_map = {
    "engineering": "Engineering",
    # "data science": "Data Science",
    # ...
}
df_clean["program"] = df_clean["program"].str.strip().str.lower().map(program_map)

# 3) turn impossible values into blanks (NaN). Example:
#    df_clean.loc[df_clean["age"] > 60, "age"] = np.nan

check_my_work(df_clean)

❌ no duplicate students
❌ program has exactly 4 clean labels
❌ age is realistic (15–60 or blank)
❌ study_hours is realistic (0–100 or blank)
❌ sleep_hours is realistic (0–14 or blank)
❌ exam_score is between 0 and 100 (or blank)
✅ real extreme students were KEPT
✅ you did not delete too much (≥ 505 rows left)

2/8 checks passed  — keep going


## Step 5 · 🌶️ Extra challenge (if you finish early)
1. Write a **3-sentence data-quality memo** to the survey office: what went wrong, how big is it, what should they change in the form? *(Hint: a dropdown for `program` would prevent label chaos.)*
2. Compare `df_clean.describe()` with `raw.describe()`. How did the mean of `study_hours_week` change?
3. **Why did we blank impossible values instead of deleting the whole row?**

**When time is up: type your final number of problems found in chat.** 🏁